# CAND-VB2 RAG — Google Colab

RAG chuyên biệt về **VB2CA tuyển mới** dành cho công dân đã có bằng đại học, ưu tiên trường hợp văn bằng 1 CNTT/IT.

## Cách dùng

- **Lần đầu / khi dataset hoặc quy định được cập nhật:** chạy cell `FULL UPDATE / BUILD` bên dưới. Index được lưu bền vững tại `MyDrive/CAND_VB2_RAG/index`.
- **Những lần chat sau:** chỉ cần chạy **CELL CUỐI — QUICK RESUME & CHAT**. Cell cuối tự mount Drive, clone/update code, đọc `/content/providers.env`, nạp index cũ và mở Gradio. **Không crawl và không embedding lại.**
- Gemini + OpenRouter là API model nên không có weight/checkpoint local để lưu. Phần được persist là knowledge index + metadata/version manifest.
- `providers.env` không bao giờ được commit lên GitHub. Ở runtime mới, nếu `/content/providers.env` chưa có, cell sẽ yêu cầu bạn upload.


## FULL UPDATE / BUILD — chỉ chạy khi dữ liệu thay đổi

Cell này xác minh cấu trúc nguồn, build index từ `data/corpus/` và lưu sang Google Drive. Nếu fingerprint corpus không đổi, script tự bỏ qua re-embedding.


In [ ]:
from pathlib import Path
import os, sys, shutil, subprocess
from google.colab import drive, files

REPO_URL = 'https://github.com/NVTruong473/NLP.git'
REPO_DIR = Path('/content/NLP')
ENV_PATH = Path('/content/providers.env')
DRIVE_ROOT = Path('/content/drive/MyDrive/CAND_VB2_RAG')
INDEX_DIR = DRIVE_ROOT / 'index'

drive.mount('/content/drive', force_remount=False)
DRIVE_ROOT.mkdir(parents=True, exist_ok=True)

if (REPO_DIR / '.git').exists():
    subprocess.run(['git', '-C', str(REPO_DIR), 'pull', '--ff-only'], check=True)
else:
    subprocess.run(['git', 'clone', REPO_URL, str(REPO_DIR)], check=True)
os.chdir(REPO_DIR)

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements.txt'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', '.', '--no-deps'], check=True)

if not ENV_PATH.exists():
    print('Upload providers.env -> /content/providers.env')
    uploaded = files.upload()
    if not uploaded:
        raise RuntimeError('Thiếu providers.env')
    selected = next(iter(uploaded))
    Path(selected).replace(ENV_PATH)

os.environ['VIETRAG_INDEX_DIR'] = str(INDEX_DIR)
os.environ['VIETRAG_ENV'] = str(ENV_PATH)

subprocess.run([sys.executable, 'scripts/validate_sources.py'], check=True)
subprocess.run([
    sys.executable, 'scripts/build_index.py',
    '--input-dir', 'data/corpus',
    '--env', str(ENV_PATH),
], check=True)

manifest = INDEX_DIR / 'index_manifest.json'
print('\nFULL BUILD READY')
print('Persistent index:', INDEX_DIR)
print('Manifest:', manifest)
print('Từ lần sau, chỉ chạy CELL CUỐI để chat.')


## CELL CUỐI — QUICK RESUME & CHAT

**Đây là cell duy nhất cần chạy ở những lần sử dụng sau.** Nó tuyệt đối không gọi `build_index.py`. Nếu Drive chưa có index, cell sẽ dừng và yêu cầu chạy FULL BUILD một lần thay vì âm thầm tốn quota embedding.


In [ ]:
# QUICK RESUME: fresh Colab runtime -> one cell -> chat
from pathlib import Path
import os, sys, subprocess, importlib.util
from google.colab import drive, files

REPO_URL = 'https://github.com/NVTruong473/NLP.git'
REPO_DIR = Path('/content/NLP')
ENV_PATH = Path('/content/providers.env')
INDEX_DIR = Path('/content/drive/MyDrive/CAND_VB2_RAG/index')

# 1) Persistent knowledge index
drive.mount('/content/drive', force_remount=False)
required_index = [INDEX_DIR/'faiss.index', INDEX_DIR/'chunks.jsonl', INDEX_DIR/'index_manifest.json']
if not all(p.exists() for p in required_index):
    missing = [str(p) for p in required_index if not p.exists()]
    raise RuntimeError('Chưa có persistent index. Hãy chạy FULL UPDATE / BUILD một lần. Missing: ' + ', '.join(missing))

# 2) Current source code (no data rebuild)
if (REPO_DIR/'.git').exists():
    subprocess.run(['git', '-C', str(REPO_DIR), 'pull', '--ff-only'], check=True)
else:
    subprocess.run(['git', 'clone', REPO_URL, str(REPO_DIR)], check=True)
os.chdir(REPO_DIR)

# Install runtime packages only when this fresh Colab does not already have them.
needed = ['faiss', 'gradio', 'rank_bm25', 'dotenv']
if any(importlib.util.find_spec(name) is None for name in needed):
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements.txt'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', '.', '--no-deps'], check=True)

# 3) Private API keys stay only in /content for this runtime.
if not ENV_PATH.exists():
    print('Không thấy /content/providers.env. Upload providers.env của bạn:')
    uploaded = files.upload()
    if not uploaded:
        raise RuntimeError('Thiếu providers.env')
    selected = next(iter(uploaded))
    Path(selected).replace(ENV_PATH)

# 4) Point the app to the SAVED index. No crawling, no embedding, no rebuild.
os.environ['VIETRAG_INDEX_DIR'] = str(INDEX_DIR)
os.environ['VIETRAG_ENV'] = str(ENV_PATH)

import json
manifest = json.loads((INDEX_DIR/'index_manifest.json').read_text(encoding='utf-8'))
print('Loaded saved index:', INDEX_DIR)
print('Built at:', manifest.get('created_at_utc'))
print('Chunks:', manifest.get('chunk_count'), '| Sources:', len(manifest.get('source_ids', [])))
print('Fingerprint:', str(manifest.get('fingerprint', ''))[:16])
print('Starting chat — NO RE-EMBEDDING...')

# New process guarantees current pulled code is loaded cleanly.
subprocess.run([sys.executable, 'app.py'], check=True, env=os.environ.copy())
